In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import FormatStrFormatter

In [2]:
main = pd.read_csv('./results.csv')

In [3]:
main['order'] = main['input'].apply(lambda x: re.findall(r"_([a-zA-Z]*).txt", x)[0])

In [4]:
main.columns

Index(['algorithm', 'M', 'input', 'N', 'k', 'rowCmp', 'ovcDecisions', 'colCmp',
       'timeInUS', 'order'],
      dtype='object')

In [11]:
def _measureXByN(algorithms: [str], k: int, measure: str, ylabel: str, order: str = "random"):
    df = main.loc[
        main['algorithm'].isin(algorithms)]
    df = df.loc[df['order']==order]
    df = df.loc[df['k'] == k]
    df = pd.pivot_table(df, values=[measure], index=['N'], columns=['algorithm'], aggfunc=['mean'])
    df = df['mean'][measure]
    df = df[algorithms]

    ax = df.plot(
            style=['grayscale'], 
            kind='bar', 
            logy=True, 
            fontsize=14,
            figsize=(8,6)
            )
    ax.tick_params(axis='x', labelrotation=0)
    plt.xlabel('N', fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    ax.get_yaxis().set_major_formatter('{x:,.0f}')
    labels = ax.get_xaxis().get_majorticklabels()
    plt.legend(df.columns, fontsize=11, loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2)


def _measureXByK(algorithms: [str], N: int, measure: str, ylabel: str):
    df = main.loc[
    main['algorithm']\
        .isin(algorithms)]
    df = df.loc[df['N'] == N]
    df = pd.pivot_table(df, values=[measure], index=['k'], columns=['algorithm'], aggfunc=['mean'])
    df = df['mean'][measure]
    df = df[algorithms]

    ax = df.plot(
            style=['grayscale'], 
            kind='bar', 
            logy=False, 
            fontsize=14,
            figsize=(8,6)
            )
    ax.tick_params(axis='x', labelrotation=0)
    plt.xlabel('Key Length', fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    ax.get_yaxis().set_major_formatter('{x:,.0f}')
    plt.legend(df.columns, fontsize=11, loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2)


def measureColCmpByN(algorithms: [str], k: int):
    _measureXByN(algorithms, k, "colCmp", 'Number of Column Comparisons')

def measureRowCmpByN(algorithms: [str], k: int):
    _measureXByN(algorithms, k, "rowCmp", 'Number of Row Comparisons')
    
def measureRuntimeByN(algorithms: [str], k: int):
    _measureXByN(algorithms, k, "timeInUS", 'Runtime in Microseconds')


def measureColCmpByK(algorithms: [str], N: int):
    _measureXByK(algorithms, N, "colCmp", 'Number of Column Comparisons')

def measureRowCmpByK(algorithms: [str], N: int):
    _measureXByK(algorithms, N, "rowCmp", 'Number of Row Comparisons')
    
def measureRuntimeByK(algorithms: [str], N: int):
    _measureXByK(algorithms, N, "timeInUS", 'Runtime in Microseconds')


In [18]:
algorithms = ['Quicksort', 'QuicksortOVC', 'QuicksortAOVC']
order = 'random'
group = 'quicksort'

def generate_graphs(algorithms: [str], order: str, fix_k: int, fix_N:int, group: str):
    measureRowCmpByN(algorithms, k=fix_k)
    plt.savefig(f"out/{group}_{order}_measureRowCmpByN_k_{fix_k}.pdf", format="pdf", bbox_inches="tight")
    plt.close()
    measureColCmpByN(algorithms, k=fix_k)
    plt.savefig(f"out/{group}_{order}_measureColCmpByN_k_{fix_k}.pdf", format="pdf", bbox_inches="tight")
    plt.close()
    measureRuntimeByN(algorithms, k=fix_k)
    plt.savefig(f"out/{group}_{order}_measureRuntimeByN_k_{fix_k}.pdf", format="pdf", bbox_inches="tight")
    plt.close()
    measureRowCmpByK(algorithms, N=fix_N)
    plt.savefig(f"out/{group}_{order}_measureRowCmpByK_N_{fix_N}.pdf", format="pdf", bbox_inches="tight")
    plt.close()
    measureColCmpByK(algorithms, N=fix_N)
    plt.savefig(f"out/{group}_{order}_measureColCmpByK_N_{fix_N}.pdf", format="pdf", bbox_inches="tight")
    plt.close()
    measureRuntimeByK(algorithms, N=fix_N)
    plt.savefig(f"out/{group}_{order}_measureRuntimeByK_N_{fix_N}.pdf", format="pdf", bbox_inches="tight")
    plt.close()

generate_graphs(algorithms, order, 20, 1000000, "quicksort")
generate_graphs(['MergesortOVC', 'QuicksortOVC', 'QuicksortAOVC', 'QuicksortAOVC+FixedAOVC', 'HeapsortOVC'], order, 20, 1000000, "summary")